In [1]:
from sklearn.datasets import fetch_california_housing 
import pandas as pd

housing = fetch_california_housing(data_home=None, download_if_missing=True, return_X_y=False, as_frame=False) # 获取加州房价数据集
X = pd.DataFrame(data=housing.data, index=None, columns=housing.feature_names, dtype=None, copy=None)
Y = pd.DataFrame(data=housing.target, index=None, columns=housing.target_names, dtype=None, copy=None)

划分数据集——训练集:测试集 = 7:3

In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, train_size=0.7, random_state=42, shuffle=True, stratify=None)

提取训练集与测试集的样本量与特征数。其中应满足
+ n_X_train = n_X_test = n_Y_train = n_Y_test
+ p_X_train = p_X_test ， p_Y_train = p_Y_test

In [3]:
n_X_train, p_X_train, n_X_test, p_X_test = X_train.shape[0], X_train.shape[1], X_test.shape[0], X_test.shape[1] 
n_Y_train, p_Y_train, n_Y_test, p_Y_test = Y_train.shape[0], Y_train.shape[1], Y_test.shape[0], Y_test.shape[1]

标准化并防止数据泄露

In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler(copy=True, with_mean=True, with_std=True) # 标准化器/模型
# scaler.fit(X=X_train, y=None, sample_weight=None) # 训练标准化模型
# X_train_standard = scaler.transform(X=X_train, y=None) # 应用模型
X_train_standard = scaler.fit_transform(X=X_train, y=None) # 与上面两行效果相同，训练标准化模型并应用
X_test_standard = scaler.transform(X=X_test, copy=None)

# 线性回归模型

In [5]:
from sklearn.linear_model import LinearRegression
import numpy as np
from sklearn.metrics import (r2_score, explained_variance_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, 
                             median_absolute_error, max_error, mean_squared_log_error, 
                             mean_poisson_deviance, mean_gamma_deviance, mean_tweedie_deviance)

lrm = LinearRegression(fit_intercept=True, copy_X=True, n_jobs=None, positive=False) # 线性回归模型
lrm.fit(X=X_train_standard, y=Y_train, sample_weight=None) # 训练线性回归模型
parameters = pd.DataFrame(data=[np.append(lrm.coef_, lrm.intercept_, axis=None)],
                          index=None, columns=list(housing.feature_names)+['intercept'], dtype=None, copy=None)
print(f"线性回归模型训练集参数拟合结果:\n{parameters}")

# 预测
Y_train_pre = lrm.predict(X=X_train_standard)
Y_test_pre = lrm.predict(X=X_test_standard)

# 评估模型结果
def regression_evaluate(y_true, y_pred, p_x):
    R2 = r2_score(y_true=y_true, y_pred=y_pred, sample_weight=None, multioutput='uniform_average', force_finite=True) # 拟合优度（R²）
    n = len(y_true)
    if n > p_x+1:
        Adjusted_R2 = 1 - (1-R2)*(n-1)/(n-p_x-1) # 调整后的拟合优度（调整后的 R²）
    else:
        Adjusted_R2 = np.nan
    # 解释方差分数（类似R²）
    EVS = explained_variance_score(y_true=y_true, y_pred=y_pred, sample_weight=None, multioutput='uniform_average', force_finite=True) 
    MSE = mean_squared_error(y_true=y_true, y_pred=y_pred, sample_weight=None, multioutput='uniform_average') # 均方误差（MSE）
    RMSE = np.sqrt(MSE) # 均方根误差（RMSE）
    MAE = mean_absolute_error(y_true=y_true, y_pred=y_pred, sample_weight=None, multioutput='uniform_average') # 平均绝对误差（MAE）
    MAPE = mean_absolute_percentage_error(y_true=y_true, y_pred=y_pred, sample_weight=None, multioutput='uniform_average') # 平均绝对百分比误差（MAPE）
    MedAE = median_absolute_error(y_true=y_true, y_pred=y_pred, sample_weight=None, multioutput='uniform_average') # 中位数绝对误差（MedAE）
    ME = max_error(y_true=y_true, y_pred=y_pred) # 最大残差（ME）
    if np.any(y_true < 0) or np.any(y_pred < 0):
        MSLE = np.nan
    else:
        MSLE = mean_squared_log_error(y_true=y_true, y_pred=y_pred, sample_weight=None, multioutput='uniform_average') # 均方对数误差（MSLE）
    
    if np.any(y_true < 0) or np.any(y_pred <= 0):
        MPD = np.nan
    else:
        MPD = mean_poisson_deviance(y_true=y_true, y_pred=y_pred, sample_weight=None) # 平均泊松偏差（MPD）
    
    if np.any(y_true <= 0) or np.any(y_pred <= 0):
        MGD = np.nan
        MTD = np.nan
    else:
        MGD = mean_gamma_deviance(y_true=y_true, y_pred=y_pred, sample_weight=None) # 平均伽马偏差（MGD）
        MTD = mean_tweedie_deviance(y_true=y_true, y_pred=y_pred, sample_weight=None, power=3) # 平均 Tweedie 偏差（MTD）

    regression_evaluation = {"R^2" : R2, 
                             "Adjusted_R^2" : Adjusted_R2, 
                             "EVS" : EVS, 
                             "MSE" : MSE, 
                             "RMSE" : RMSE, 
                             "MAE" : MAE, 
                             "MAPE" : MAPE,
                             "MedAE" : MedAE,
                             "ME" : ME,
                             "MSLE" : MSLE,
                             "MPD" : MPD,
                             "MGD" : MGD,
                             "MTD" : MTD
                             }

    return pd.DataFrame(data=[regression_evaluation], index=["VALUE"], columns=None, dtype=None, copy=None)

evaluation_train = regression_evaluate(y_true=Y_train, y_pred=Y_train_pre, p_x=p_X_train)
evaluation_test = regression_evaluate(y_true=Y_test, y_pred=Y_test_pre, p_x=p_X_test)
print(f"训练集评估结果：\n{evaluation_train} \n测试集评估结果：\n{evaluation_test}")

线性回归模型训练集参数拟合结果:
     MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  0.849186   0.12197 -0.299346   0.348218   -0.001004 -0.041681 -0.893844   

   Longitude  intercept  
0  -0.868475   2.069158  
训练集评估结果：
            R^2  Adjusted_R^2       EVS       MSE      RMSE       MAE  \
VALUE  0.609367       0.60915  0.609367  0.523329  0.723415  0.530922   

           MAPE     MedAE        ME  MSLE  MPD  MGD  MTD  
VALUE  0.316006  0.412956  5.905949   NaN  NaN  NaN  NaN   
测试集评估结果：
            R^2  Adjusted_R^2       EVS       MSE      RMSE       MAE  \
VALUE  0.595781      0.595258  0.595782  0.530553  0.728391  0.527231   

           MAPE     MedAE        ME  MSLE  MPD  MGD  MTD  
VALUE  0.317473  0.409091  9.878569   NaN  NaN  NaN  NaN  
